### Решение состоит из двух этапов: сначала несколько retrieval-моделей формируют широкий набор кандидатов, затем CatBoostRanker переупорядочивает их и выбирает итоговые top-50 объявлений

## 1. Импорты и настройки

In [1]:
from pathlib import Path
import gc
import re

import faiss
import numpy as np
import pandas as pd
from IPython.display import display
from catboost import CatBoostRanker
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

DATA_DIR = Path('.').resolve()
CACHE_DIR = DATA_DIR / 'e5_cache'
MODEL_DIR = DATA_DIR
MODEL_PATH = MODEL_DIR / 'yeti_full.cbm'
OUTPUT_PATH = DATA_DIR / 'answer.csv'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
CHANNEL_TOP_K = {
    'bm25_combined': 1500,
    'dense': 2000,
    'tfidf_word': 1200,
    'bm25_title': 700,
    'bm25_description': 300,
    'bm25_params': 300, 
}  # Размер пулов отдельных каналов
DENSE_TOP_K = 2000
CANDIDATE_K = 2000
TOP_K = 50
TRAIN_CHUNK_SIZE = 10_000
INFERENCE_CHUNK_SIZE = 128
E5_MODEL_NAME = 'intfloat/multilingual-e5-base' 
E5_BATCH_SIZE = 128
E5_MAX_LENGTH = 128

FEATURES = ['location', 'params', 'quality'] # Основные группы фичей, которые идут на вход ранкера катбуст
RANK_CHANNELS = [
    'bm25_title', 'bm25_params', 'bm25_description',
    'bm25_combined', 'tfidf_word', 'dense',
]
QUERY_COLS = [
    'search_query', 'search_location_id', 'search_is_delivery_search',
    'search_infm_params_text', 'search_category',
]
ITEM_COLS = [
    'item_id', 'item_title_raw', 'item_infm_params_text',
    'item_description_raw', 'item_location_id', 'item_rating',
    'item_rating_reviews_count',
]

## 2. Загрузка данных

In [2]:
train_pairs = pd.read_parquet(DATA_DIR / 'train.parquet', columns=QUERY_COLS + ['item_id'])
train_pairs['query_uid'] = pd.util.hash_pandas_object(
    train_pairs[QUERY_COLS], index=False
).astype(str)
train_queries = train_pairs.drop_duplicates('query_uid')[['query_uid'] + QUERY_COLS].reset_index(drop=True)
train_labels = train_pairs.groupby('query_uid')['item_id'].agg(frozenset).to_dict()
train_items = pd.read_parquet(
    DATA_DIR / 'train.parquet', columns=ITEM_COLS
).drop_duplicates('item_id').reset_index(drop=True)
benchmark_queries = pd.read_parquet(DATA_DIR / 'benchmark_queries.parquet')
benchmark_items = pd.read_parquet(
    DATA_DIR / 'benchmark_items.parquet', columns=ITEM_COLS
).drop_duplicates('item_id').reset_index(drop=True)

print(f'Train: {len(train_queries):,} запросов, {len(train_items):,} объявлений')
print(f'Benchmark: {len(benchmark_queries):,} запросов, {len(benchmark_items):,} объявлений')

Train: 354,463 запросов, 344,825 объявлений
Benchmark: 2,452 запросов, 189,212 объявлений


## 3. Подготовка текстов и лексические каналы

BM25 строится отдельно по заголовку, параметрам, описанию и объединенному
тексту; еще один канал использует word TF-IDF. В объединенном тексте
заголовок, параметры и описание берутся с разными весами, подобранными опытным путем

In [14]:
def normalize_text(series):
    return series.fillna('').astype(str).str.lower().str.replace('ё', 'е')


def field_queries(df):
    query = normalize_text(df['search_query'])
    params = normalize_text(df['search_infm_params_text'])
    combined = query + ' ' + params
    return {
        'title': query.tolist(),
        'params': combined.tolist(),
        'description': combined.tolist(),
        'combined': (query + ' ' + query + ' ' + params).tolist(),
        'word_tfidf': combined.tolist(),
    }


class BM25Retriever:
    def __init__(self, k1=1.5, b=0.75, min_df=2, max_features=250_000):
        self.k1 = k1
        self.b = b
        self.vectorizer = CountVectorizer(
            min_df=min_df,
            max_features=max_features,
            dtype=np.float32,
        )

    def fit(self, documents, item_ids):
        counts = self.vectorizer.fit_transform(documents).tocoo()
        n_docs = counts.shape[0]
        doc_len = np.asarray(counts.tocsr().sum(axis=1)).ravel()
        df = np.bincount(counts.col, minlength=counts.shape[1])
        idf = np.log1p((n_docs - df + 0.5) / (df + 0.5))
        norm = self.k1 * (1 - self.b + self.b * doc_len / doc_len.mean())

        counts.data *= (self.k1 + 1) / (counts.data + norm[counts.row])
        counts.data *= idf[counts.col]
        self.matrix = counts.tocsr()
        self.item_ids = np.asarray(item_ids)
        return self

    def search(self, queries, top_k=300, batch_size=64):
        results = []
        for start in range(0, len(queries), batch_size):
            query_matrix = self.vectorizer.transform(queries[start:start + batch_size])
            query_matrix.data[:] = 1.0
            scores = (query_matrix @ self.matrix.T).tocsr()

            for row in range(scores.shape[0]):
                left, right = scores.indptr[row:row + 2]
                values = scores.data[left:right]
                indices = scores.indices[left:right]
                if len(values) > top_k:
                    chosen = np.argpartition(values, -top_k)[-top_k:]
                    chosen = chosen[np.argsort(-values[chosen])]
                else:
                    chosen = np.argsort(-values)
                results.append(list(zip(
                    self.item_ids[indices[chosen]], values[chosen].astype(float)
                )))
        return results


class TfidfRetriever:
    def __init__(self, analyzer='word', ngram_range=(1, 1), min_df=2, max_features=250_000):
        self.vectorizer = TfidfVectorizer(
            analyzer=analyzer, ngram_range=ngram_range, min_df=min_df,
            max_features=max_features, sublinear_tf=True, dtype=np.float32,
        )

    def fit(self, documents, item_ids):
        self.matrix = self.vectorizer.fit_transform(documents)
        self.item_ids = np.asarray(item_ids)
        return self

    def search(self, queries, top_k=300, batch_size=64):
        results = []
        for start in range(0, len(queries), batch_size):
            query_matrix = self.vectorizer.transform(queries[start:start + batch_size])
            scores = (query_matrix @ self.matrix.T).tocsr()
            for row in range(scores.shape[0]):
                left, right = scores.indptr[row:row + 2]
                values = scores.data[left:right]
                indices = scores.indices[left:right]
                if len(values) > top_k:
                    chosen = np.argpartition(values, -top_k)[-top_k:]
                    chosen = chosen[np.argsort(-values[chosen])]
                else:
                    chosen = np.argsort(-values)
                results.append(list(zip(self.item_ids[indices[chosen]], values[chosen].astype(float))))
        return results


def combined_item_text(items):
    title = normalize_text(items['item_title_raw'])
    params = normalize_text(items['item_infm_params_text'])
    description = normalize_text(items['item_description_raw']).str.slice(0, 1_000)
    return title + ' ' + title + ' ' + title + ' ' + params + ' ' + params + ' ' + description


def build_channel_indices(items): # подсчет скора для объявления по каждому каналу
    text_columns = {
        'bm25_title': ('item_title_raw', BM25Retriever()),
        'bm25_params': ('item_infm_params_text', BM25Retriever()),
        'bm25_description': ('item_description_raw', BM25Retriever()),
        'bm25_combined': (None, BM25Retriever()),
        'tfidf_word': (None, TfidfRetriever(analyzer='word')),
    }
    indices = {}
    for name, (column, retriever) in text_columns.items():
        if name == 'bm25_combined':
            texts = combined_item_text(items)
        elif name == 'tfidf_word':
            texts = combined_item_text(items)
        else:
            texts = normalize_text(items[column])
            if name == 'bm25_description':
                texts = texts.str.slice(0, 1_000)
        indices[name] = retriever.fit(texts, items['item_id'])
    return indices


def retrieve_channels_with_scores(indices, queries, top_k_by_channel): # Отбор кандидатов по каждому отдельному признаку
    query_texts = field_queries(queries)
    query_keys = {
        'bm25_title': 'title', 'bm25_params': 'params',
        'bm25_description': 'description', 'bm25_combined': 'combined',
        'tfidf_word': 'word_tfidf',
    }
    ids, scores = {}, {}
    for name, index in indices.items():
        hits = index.search(query_texts[query_keys[name]], top_k=top_k_by_channel[name])
        ids[name] = [[item_id for item_id, _ in row] for row in hits]
        scores[name] = [[float(score) for _, score in row] for row in hits]
    return ids, scores


def union_candidates(channel_results, names):   # функция объединяет кандидатов со всех информационных каналов, в итоге ее работы на каждый запрос 
    rows = []                                   # отбирается порядка 3000 первичных кандидатов
    for row in range(len(next(iter(channel_results.values())))):
        seen = set()
        merged = []
        for name in names:
            for item_id in channel_results[name][row]:
                if item_id not in seen:
                    merged.append(item_id)
                    seen.add(item_id)
        rows.append(merged)
    return rows


def token_set(text):
    text = str(text).lower().replace('ё', 'е')
    return set(re.findall(r'(?u)\b\w\w+\b', text))


def rank_maps(row, lexical_channels, dense_lists):
    channels = {name: lexical_channels[name][row] for name in lexical_channels}
    channels['dense'] = dense_lists[row]
    return {
        name: {item_id: rank for rank, item_id in enumerate(items, 1)}
        for name, items in channels.items()
    }


def sample_train_candidates(candidates, queries, labels, lexical_channels, dense_lists, seed=RANDOM_STATE): #  подготовка блока примеров данных для катбуста
    rng = np.random.default_rng(seed). 
    sampled, keep = [], []

    for row, (items, uid) in enumerate(zip(candidates, queries['query_uid'])):
        positives = [x for x in items if x in labels[uid]]
        if not positives:
            continue

        ranks = rank_maps(row, lexical_channels, dense_lists)
        negatives = [x for x in items if x not in labels[uid]]

        negatives.sort(
            key=lambda x: min(ranks[name].get(x, np.inf) for name in RANK_CHANNELS)
        )

        hard = negatives[:15] # для обучения ранкера используются все положительные примеры, 15 строго отрицательных и еще несколько случаных
        remaining = negatives[15:]
        random_neg = rng.choice(
            remaining, size=min(5, len(remaining)), replace=False
        ).tolist()

        sampled.append(positives + hard + random_neg)
        keep.append(row)

    return sampled, np.asarray(keep)

## 4. E5 embeddings и кэш

Используем `intfloat/multilingual-e5-base`, префиксы `query:`/`passage:`,
`max_length=128` и нормализованные embeddings. При наличии
кэша проверяется точный порядок идентификаторов. 

In [4]:
def e5_item_text(row):
    return (
        f"passage: {str(row['item_title_raw']).lower()} "
        f"{str(row['item_infm_params_text']).lower()}"
    )


def e5_query_text(row):
    return (
        f"query: {str(row['search_query']).lower()} "
        f"{str(row['search_infm_params_text']).lower()}"
    )


def load_or_create_embeddings(name, frame, id_column, text_function):
    path = CACHE_DIR / f'{name}_e5.npy'
    ids_path = CACHE_DIR / f'{name}_ids.npy'
    expected_ids = frame[id_column].astype(str).to_numpy()

    def checked(embeddings, cached_ids):
        if embeddings.shape != (len(frame), 768):
            raise ValueError(f'Размерность кэша {name} не совпадает с данными: {embeddings.shape}')
        if not np.array_equal(cached_ids.astype(str), expected_ids):
            raise ValueError(f'Порядок ID в кэше {name} не совпадает с DataFrame')
        return embeddings

    if path.exists():
        if not ids_path.exists():
            raise ValueError(f'Для кэша {path.name} нет файла ID {ids_path.name}')
        embeddings = np.load(path, mmap_mode='r')
        cached_ids = np.load(ids_path, allow_pickle=False)
        print(f'Загружен кэш: {path.name}')
        return checked(embeddings, cached_ids)

    part_files = [
        p for p in CACHE_DIR.glob('*.npy')
        if p.stem.startswith(name) and 'ids' not in p.stem
        and re.search(r'(?:part|chunk)[_-]?(\d+)$', p.stem)
    ]
    if part_files:
        part_files.sort(key=lambda p: int(re.search(r'(\d+)$', p.stem).group(1)))
        part_ids, parts = [], []
        for part in part_files:
            part_number = re.search(r'(\d+)$', part.stem).group(1)
            matches = [
                p for p in CACHE_DIR.glob('*.npy')
                if name in p.stem and 'ids' in p.stem
                and re.search(r'(?:part|chunk)[_-]?' + part_number + r'$', p.stem)
            ]
            if len(matches) != 1:
                raise ValueError(f'Невозможно проверить порядок ID части {part.name}')
            parts.append(np.load(part))
            part_ids.append(np.load(matches[0], allow_pickle=False))
        embeddings = checked(np.concatenate(parts, axis=0), np.concatenate(part_ids))
        np.save(path, embeddings)
        np.save(ids_path, np.asarray(expected_ids, dtype='<U64'))
        print(f'Собран единый кэш: {path.name}')
        return np.load(path, mmap_mode='r')

    import torch
    from sentence_transformers import SentenceTransformer
    device = 'mps' if torch.backends.mps.is_available() else 'cpu'
    model_e5 = SentenceTransformer(E5_MODEL_NAME, device=device)
    model_e5.max_seq_length = E5_MAX_LENGTH
    texts = frame.apply(text_function, axis=1).tolist()
    print(f'Расчёт embeddings {name}: {len(texts):,} текстов')
    embeddings = model_e5.encode(
        texts, batch_size=E5_BATCH_SIZE, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    )
    np.save(path, embeddings)
    np.save(ids_path, np.asarray(expected_ids, dtype='<U64'))
    return checked(np.load(path, mmap_mode='r'), np.load(ids_path, allow_pickle=False))


train_item_embeddings = load_or_create_embeddings(
    'train_items', train_items, 'item_id', e5_item_text
)
train_query_embeddings = load_or_create_embeddings(
    'train_queries', train_queries, 'query_uid', e5_query_text
)
benchmark_item_embeddings = load_or_create_embeddings(
    'benchmark_items', benchmark_items, 'item_id', e5_item_text
)
benchmark_query_embeddings = load_or_create_embeddings(
    'benchmark_queries', benchmark_queries, 'query_id', e5_query_text
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Расчёт embeddings train_items: 344,825 текстов


Batches:   0%|          | 0/2694 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Расчёт embeddings benchmark_queries: 2,452 текстов


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

## 5. Индексы retrieval

Для каждого набора объявлений заранее строятся поисковые индексы, которые позволяют быстро получать подходящих кандидатов для каждого запроса

In [5]:
def build_indices(items, item_embeddings):
    lexical_indices = build_channel_indices(items)
    dense_index = faiss.IndexFlatIP(item_embeddings.shape[1])
    dense_index.add(np.asarray(item_embeddings, dtype='float32'))
    return lexical_indices, dense_index

## 6. Получение и объединение кандидатов

Каждый канал возвращает свое top-K. Для каждого запроса результаты всех retrieval-каналов объединяются в единый список кандидатов без дублей. Если объявление найдено несколькими каналами, в общем списке оно остается один раз.

In [6]:
def retrieve_for_queries(queries, query_embeddings, items, lexical_indices, dense_index):
    lexical, lexical_scores = retrieve_channels_with_scores(
        lexical_indices, queries, top_k_by_channel=CHANNEL_TOP_K
    )
    dense_scores_array, positions = dense_index.search(
        np.asarray(query_embeddings, dtype='float32'), DENSE_TOP_K
    )
    item_ids = items['item_id'].to_numpy()
    dense = [[item_ids[i] for i in row if i >= 0] for row in positions]
    dense_scores = [
        [float(score) for score, i in zip(score_row, row) if i >= 0]
        for score_row, row in zip(dense_scores_array, positions)
    ]
    union = union_candidates({**lexical, 'dense': dense}, list(lexical) + ['dense'])
    return lexical, lexical_scores, dense, dense_scores, union

## 7. Девять признаков CatBoost

Обычные признаки: совпадение локации, пересечение параметров и quality.
Quality объединяет рейтинг и логарифм числа отзывов. Шесть остальных
признаков — позиции объявления в retrieval каналах; отсутствие в канале
обозначается рангом `2001`. Масштаб отзывов рассчитан по train объявлениям
и применяется ко всем корпусам.

In [7]:
review_scale = max(
    np.log1p(train_items['item_rating_reviews_count'].fillna(0)).quantile(0.99),
    1.0,
)


def prepare_items(items):
    item_table = items.set_index('item_id')
    item_params = normalize_text(items['item_infm_params_text'])
    params_tokens = dict(zip(items['item_id'], item_params.map(token_set)))
    return item_table, params_tokens


def build_pair_features(candidates, queries, item_table, params_tokens):
    query_params = normalize_text(queries['search_infm_params_text']).map(token_set).tolist()
    feature_rows = []
    for row, item_ids in enumerate(candidates):
        selected = item_table.loc[item_ids]
        params = [
            len(query_params[row] & params_tokens[item_id]) / max(len(query_params[row]), 1)
            for item_id in item_ids
        ]
        rating = selected['item_rating'].fillna(0).clip(0, 5).to_numpy() / 5
        reviews = np.log1p(
            selected['item_rating_reviews_count'].fillna(0).to_numpy()
        ) / review_scale
        feature_rows.append({
            'location': (
                selected['item_location_id'].to_numpy()
                == queries.iloc[row]['search_location_id']
            ).astype(np.float32) * (1 - queries.iloc[row]['search_is_delivery_search']),
            'params': np.asarray(params, dtype=np.float32),
            'quality': 0.7 * rating + 0.3 * np.clip(reviews, 0, 1),
        })
    return feature_rows


def make_feature_matrix(feature_rows, candidates, queries, lexical, dense):
    rows = []
    for row, (uid, item_ids, features) in enumerate(
        zip(queries['query_uid'], candidates, feature_rows)
    ):
        ranks = rank_maps(row, lexical, dense)
        for i, item_id in enumerate(item_ids):
            rows.append(
                [features[name][i] for name in FEATURES]
                + [ranks[name].get(item_id, CANDIDATE_K + 1) for name in RANK_CHANNELS]
            )
    matrix = np.asarray(rows, dtype=np.float32)
    assert matrix.ndim == 2 and matrix.shape[1] == 9
    assert np.isfinite(matrix).all()
    return matrix


def rank_top50(candidates, scores):
    predictions, offset = [], 0
    for items in candidates:
        order = np.argsort(-scores[offset:offset + len(items)], kind='stable')
        predictions.append([items[i] for i in order[:TOP_K]])
        offset += len(items)
    assert offset == len(scores)
    return predictions

## 8. Разделение train и validation

Для локальной оценки из train случайно выбираются 250 000 уникальных запросов и делятся в пропорции 90/10 на обучение и validation. Для validation используется полный естественный candidate pool без искусственного добавления положительных объявлений. После оценки финальная модель обучается уже на всех размеченных запросаx

In [8]:
def make_training_split():
    rng = np.random.default_rng(RANDOM_STATE)
    selected_uids = rng.choice(
        train_queries['query_uid'].to_numpy(), size=250_000, replace=False
    )
    selected_mask = train_queries['query_uid'].isin(selected_uids).to_numpy()
    selected_queries = train_queries[selected_mask].reset_index(drop=True)
    selected_positions = np.flatnonzero(selected_mask)
    group_train, group_valid = train_test_split(
        selected_queries['query_uid'].to_numpy(),
        test_size=0.10, random_state=RANDOM_STATE,
    )
    train_uids, valid_uids = set(group_train), set(group_valid)
    train_mask = selected_queries['query_uid'].isin(train_uids).to_numpy()
    valid_mask = selected_queries['query_uid'].isin(valid_uids).to_numpy()
    return (
        selected_queries[train_mask].reset_index(drop=True),
        np.asarray(train_query_embeddings[selected_positions[train_mask]]),
        selected_queries[valid_mask].reset_index(drop=True),
        np.asarray(train_query_embeddings[selected_positions[valid_mask]]),
        train_uids,
        valid_uids,
    )

## 9. Обучающие пары по частям

На каждый train запрос берем найденные positives, 15 сильных и 5 случайных
negatives. Запрос без найденного positive пропускается. Retrieval и признаки
считаются чанками по 10 000 запросов, индексы и подготовленные item данные
переиспользуются.

In [9]:
def build_training_matrix(queries, embeddings, labels, items,
                          lexical_indices, dense_index, item_table, params_tokens):
    xs, ys, groups = [], [], []
    for start in range(0, len(queries), TRAIN_CHUNK_SIZE):
        stop = min(start + TRAIN_CHUNK_SIZE, len(queries))
        query_chunk = queries.iloc[start:stop].reset_index(drop=True)
        embedding_chunk = embeddings[start:stop]
        channels, _, dense, _, candidates = retrieve_for_queries(
            query_chunk, embedding_chunk, items, lexical_indices, dense_index
        )
        sampled, keep = sample_train_candidates(
            candidates, query_chunk, labels, channels, dense
        )
        if not len(keep):
            continue
        sampled_queries = query_chunk.iloc[keep].reset_index(drop=True)
        sampled_channels = {
            name: [rows[i] for i in keep] for name, rows in channels.items()
        }
        sampled_dense = [dense[i] for i in keep]
        sampled_features = build_pair_features(
            sampled, sampled_queries, item_table, params_tokens
        )
        x = make_feature_matrix(
            sampled_features, sampled, sampled_queries,
            sampled_channels, sampled_dense,
        )
        y = np.asarray([
            int(item_id in labels[uid])
            for uid, item_ids in zip(sampled_queries['query_uid'], sampled)
            for item_id in item_ids
        ], dtype=np.float32)
        g = np.repeat(sampled_queries['query_uid'].to_numpy(), [len(x) for x in sampled])
        xs.append(x)
        ys.append(y)
        groups.append(g)
        print(f'Обработаны запросы {start:,}–{stop:,}')
        del channels, dense, candidates, sampled_features
    return np.concatenate(xs), np.concatenate(ys), np.concatenate(groups)

## 10. Загрузка или обучение CatBoostRanker

Готовый `yeti_full.cbm` загружается сразу. Если файл отсутствует, повторяем
конфигурацию финальной модели: 1000 деревьев, глубина 5, learning rate 0.05,
`YetiRank`, `NDCG:top=50`.

In [10]:
def new_ranker():
    return CatBoostRanker(
        iterations=1000, depth=5, learning_rate=0.05,
        loss_function='YetiRank', eval_metric='NDCG:top=50',
        random_seed=RANDOM_STATE, verbose=100,
    )


trained_now = False
if MODEL_PATH.exists():
    model = CatBoostRanker()
    model.load_model(MODEL_PATH)
    print(f'Загружена модель: {MODEL_PATH.name}')
else:
    trained_now = True
    train_lexical, train_dense_index = build_indices(train_items, train_item_embeddings)
    train_item_table, train_params_tokens = prepare_items(train_items)
    cb_train_queries, cb_train_embeddings, valid_queries, valid_embeddings, train_uids, valid_uids = make_training_split()
    cb_train_labels = {uid: train_labels[uid] for uid in train_uids}
    valid_labels = {uid: train_labels[uid] for uid in valid_uids}

    train_x, train_y, train_groups = build_training_matrix(
        cb_train_queries, cb_train_embeddings, cb_train_labels, train_items,
        train_lexical, train_dense_index, train_item_table, train_params_tokens,
    )
    validation_model = new_ranker()
    validation_model.fit(train_x, train_y, group_id=train_groups)

    # Оценка идёт до обучения финальной модели на всех размеченных запросах.
    valid_predictions, valid_pool_recalls = [], []
    for start in range(0, len(valid_queries), INFERENCE_CHUNK_SIZE):
        stop = min(start + INFERENCE_CHUNK_SIZE, len(valid_queries))
        query_chunk = valid_queries.iloc[start:stop].reset_index(drop=True)
        channels, _, dense, _, candidates = retrieve_for_queries(
            query_chunk, valid_embeddings[start:stop], train_items,
            train_lexical, train_dense_index,
        )
        feature_rows = build_pair_features(
            candidates, query_chunk, train_item_table, train_params_tokens
        )
        x = make_feature_matrix(feature_rows, candidates, query_chunk, channels, dense)
        valid_predictions.extend(rank_top50(candidates, validation_model.predict(x)))
        valid_pool_recalls.extend([
            len(set(items) & valid_labels[uid]) / len(valid_labels[uid])
            for items, uid in zip(candidates, query_chunk['query_uid'])
        ])
    recall_50 = np.mean([
        len(set(items) & valid_labels[uid]) / len(valid_labels[uid])
        for items, uid in zip(valid_predictions, valid_queries['query_uid'])
    ])
    hit_50 = np.mean([
        bool(set(items) & valid_labels[uid])
        for items, uid in zip(valid_predictions, valid_queries['query_uid'])
    ])
    display(pd.DataFrame([{
        'Recall@pool': np.mean(valid_pool_recalls),
        'Recall@50': recall_50,
        'HitRate@50': hit_50,
    }]))

    used_uids = train_uids | valid_uids
    extra_mask = ~train_queries['query_uid'].isin(used_uids).to_numpy()
    extra_queries = train_queries[extra_mask].reset_index(drop=True)
    extra_embeddings = np.asarray(train_query_embeddings[np.flatnonzero(extra_mask)])
    extra_labels = {uid: train_labels[uid] for uid in extra_queries['query_uid']}
    extra_x, extra_y, extra_groups = build_training_matrix(
        extra_queries, extra_embeddings, extra_labels, train_items,
        train_lexical, train_dense_index, train_item_table, train_params_tokens,
    )
    valid_train_x, valid_train_y, valid_train_groups = build_training_matrix(
        valid_queries, valid_embeddings, valid_labels, train_items,
        train_lexical, train_dense_index, train_item_table, train_params_tokens,
    )
    full_x = np.concatenate([train_x, extra_x, valid_train_x])
    full_y = np.concatenate([train_y, extra_y, valid_train_y])
    full_groups = np.concatenate([train_groups, extra_groups, valid_train_groups])
    model = new_ranker()
    model.fit(full_x, full_y, group_id=full_groups)
    model.save_model(MODEL_PATH)
    print(f'Финальная модель сохранена: {MODEL_PATH.name}')
    del train_lexical, train_dense_index, train_item_table, train_params_tokens
    gc.collect()

model_features = len(model.feature_names_)
if model_features != len(FEATURES) + len(RANK_CHANNELS):
    raise ValueError(f'Модель ожидает {model_features} признаков, код создаёт 9')

Обработаны запросы 0–10,000
Обработаны запросы 10,000–20,000
Обработаны запросы 20,000–30,000
Обработаны запросы 30,000–40,000
Обработаны запросы 40,000–50,000
Обработаны запросы 50,000–60,000
Обработаны запросы 60,000–70,000
Обработаны запросы 70,000–80,000
Обработаны запросы 80,000–90,000
Обработаны запросы 90,000–100,000
Обработаны запросы 100,000–110,000
Обработаны запросы 110,000–120,000
Обработаны запросы 120,000–130,000
Обработаны запросы 130,000–140,000
Обработаны запросы 140,000–150,000
Обработаны запросы 150,000–160,000
Обработаны запросы 160,000–170,000
Обработаны запросы 170,000–180,000
Обработаны запросы 180,000–190,000
Обработаны запросы 190,000–200,000
Обработаны запросы 200,000–210,000
Обработаны запросы 210,000–220,000
Обработаны запросы 220,000–225,000
0:	total: 473ms	remaining: 7m 52s
100:	total: 39.7s	remaining: 5m 53s
200:	total: 1m 17s	remaining: 5m 9s
300:	total: 1m 55s	remaining: 4m 29s
400:	total: 2m 33s	remaining: 3m 49s
500:	total: 3m 13s	remaining: 3m 12s
60

,Recall@pool,Recall@50,HitRate@50
0,0.857976,0.675509,0.69904


Обработаны запросы 0–10,000
Обработаны запросы 10,000–20,000
Обработаны запросы 20,000–30,000
Обработаны запросы 30,000–40,000
Обработаны запросы 40,000–50,000
Обработаны запросы 50,000–60,000
Обработаны запросы 60,000–70,000
Обработаны запросы 70,000–80,000
Обработаны запросы 80,000–90,000
Обработаны запросы 90,000–100,000
Обработаны запросы 100,000–104,463
Обработаны запросы 0–10,000
Обработаны запросы 10,000–20,000
Обработаны запросы 20,000–25,000
0:	total: 629ms	remaining: 10m 28s
100:	total: 1m 1s	remaining: 9m 4s
200:	total: 2m	remaining: 7m 58s
300:	total: 2m 58s	remaining: 6m 55s
400:	total: 3m 57s	remaining: 5m 54s
500:	total: 4m 55s	remaining: 4m 53s
600:	total: 5m 53s	remaining: 3m 54s
700:	total: 6m 51s	remaining: 2m 55s
800:	total: 7m 48s	remaining: 1m 56s
900:	total: 8m 46s	remaining: 57.8s
999:	total: 9m 43s	remaining: 0us
Финальная модель сохранена: yeti_full.cbm


## 11. Локальная оценка

При предыдущем query-disjoint split проверенная конфигурация получила
**Recall@pool = 0.857976**, **Recall@50 = 0.675509** и
**HitRate@50 = 0.69904**. Это оценка модели до финального дообучения на
всех размеченных запросах. 

## 12. Benchmark inference

In [11]:
benchmark_lexical, benchmark_dense_index = build_indices(
    benchmark_items, benchmark_item_embeddings
)
benchmark_item_table, benchmark_params_tokens = prepare_items(benchmark_items)
benchmark_predictions = []
for start in range(0, len(benchmark_queries), INFERENCE_CHUNK_SIZE):
    stop = min(start + INFERENCE_CHUNK_SIZE, len(benchmark_queries))
    query_chunk = benchmark_queries.iloc[start:stop].copy().reset_index(drop=True)
    query_chunk['query_uid'] = query_chunk['query_id'].astype(str)
    channels, _, dense, _, candidates = retrieve_for_queries(
        query_chunk, benchmark_query_embeddings[start:stop], benchmark_items,
        benchmark_lexical, benchmark_dense_index,
    )
    feature_rows = build_pair_features(
        candidates, query_chunk, benchmark_item_table, benchmark_params_tokens
    )
    x = make_feature_matrix(feature_rows, candidates, query_chunk, channels, dense)
    assert x.shape[1] == model_features
    benchmark_predictions.extend(rank_top50(candidates, model.predict(x)))
assert len(benchmark_predictions) == len(benchmark_queries)

## 13. Создание и проверка answer.csv

Проверяем состав запросов, количество объявлений, дубли и принадлежность
всех `item_id` к benchmark, затем сохраняем CSV без индекса.

In [12]:
answer = pd.DataFrame({
    'query_id': benchmark_queries['query_id'].astype(str),
    'answer': [' '.join(items) for items in benchmark_predictions],
})
item_id_set = set(benchmark_items['item_id'])
answer_lists = answer['answer'].str.split()
assert answer.columns.tolist() == ['query_id', 'answer']
assert len(answer) == len(benchmark_queries)
assert answer['query_id'].is_unique
assert set(answer['query_id']) == set(benchmark_queries['query_id'].astype(str))
assert answer['query_id'].str.fullmatch(r'.{16}').all()
assert answer_lists.map(len).le(TOP_K).all()
assert answer_lists.map(lambda x: len(x) == len(set(x))).all()
assert answer_lists.map(lambda x: set(x) <= item_id_set).all()
assert answer_lists.explode().str.fullmatch(r'[0-9a-f]{16}').all()
answer.to_csv(OUTPUT_PATH, index=False)
print(f'answer.csv сохранён: {OUTPUT_PATH}')
print(answer.shape)
display(answer.head())

answer.csv сохранён: /Users/roman666/Documents/AvitoTask/answer.csv
(2452, 2)


,query_id,answer
0,70DfDUpwjxB4lzFd,33f8d361a92bdcde fa4ba2c2cf71b0b1 75368254f36b...
1,JTrdTaZJvSiLPkXj,dfa239d031b11832 f46f68aaa4d7de38 0529307cdfbe...
2,LZCZNoVG4AFUkVRJ,a9840be7d8930cd1 35a357d7a0777087 53fc0e81b65d...
3,660ac9QVtXkRxZC3,a846a2a5241e1180 4816fc8b40866f56 bd2cb4500fad...
4,YgHcM9MVbxKnxD1e,615c73ea4c3bfd15 13f58fe9542bdb1a 53f9c8caeeb8...
